# 🔍 Reto Semana 9: SecureBank Fraud Detection

## Sistema de Detección de Anomalías en Transacciones con NumPy

---

### 📋 Contexto del Proyecto

**SecureBank** es uno de los bancos digitales más grandes de México. Has sido contratado como **Analista de Riesgos Junior** para desarrollar un sistema básico de detección de transacciones anómalas que podrían indicar fraude.

```
╔══════════════════════════════════════════════════════════════════╗
║                    SECUREBANK FRAUD DETECTION                   ║
║                Sistema de Detección de Anomalías                ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║    💳 Transacciones analizadas: 500 por categoría               ║
║    📊 Categorías: 5 tipos de comercio                           ║
║    🎯 Objetivo: Detectar transacciones sospechosas              ║
║    📈 Métodos: IQR y Z-Score                                    ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
```

### 🎯 Objetivos de Aprendizaje

En este reto aplicarás:
- Cálculo de percentiles y cuartiles
- Detección de outliers con método IQR
- Detección de outliers con Z-Score
- Análisis estadístico por categorías
- Generación de reportes de anomalías

---

## 📦 Configuración Inicial

Ejecuta esta celda para cargar NumPy y preparar el entorno.

In [1]:
import numpy as np

# Configuración para reproducibilidad
np.random.seed(2024)

# Configuración de impresión
np.set_printoptions(precision=2, suppress=True)

print("✅ NumPy cargado correctamente")
print(f"   Versión: {np.__version__}")

✅ NumPy cargado correctamente
   Versión: 2.4.6


---

## 📊 Datos del Reto

### Estructura de los Datos

```
CATEGORÍAS DE TRANSACCIONES
════════════════════════════════════════════════════════════════

Índice │ Categoría           │ Monto Típico    │ Descripción
───────┼─────────────────────┼─────────────────┼───────────────
   0   │ Supermercados       │ $200 - $2,000   │ Compras diarias
   1   │ Restaurantes        │ $100 - $800     │ Alimentos
   2   │ Gasolineras         │ $300 - $1,500   │ Combustible
   3   │ Tiendas Online      │ $150 - $5,000   │ E-commerce
   4   │ Entretenimiento     │ $50 - $500      │ Cine, streaming

════════════════════════════════════════════════════════════════

⚠️ TIPOS DE ANOMALÍAS SIMULADAS:
─────────────────────────────────────────────────────────────────
• Montos inusualmente altos (posible fraude)
• Montos inusualmente bajos (posibles pruebas de tarjeta)
• Patrones atípicos por categoría
─────────────────────────────────────────────────────────────────
```

Ejecuta la siguiente celda para generar los datos de transacciones.

In [2]:
# ═══════════════════════════════════════════════════════════════════
#                  GENERACIÓN DE DATOS DE TRANSACCIONES
# ═══════════════════════════════════════════════════════════════════

np.random.seed(2024)

# Configuración por categoría
categorias = ['Supermercados', 'Restaurantes', 'Gasolineras', 'Tiendas_Online', 'Entretenimiento']
n_categorias = len(categorias)

# Parámetros de distribución por categoría (media, desv_std)
params_categorias = {
    'Supermercados': (800, 400),
    'Restaurantes': (350, 150),
    'Gasolineras': (700, 250),
    'Tiendas_Online': (1200, 800),
    'Entretenimiento': (200, 100)
}

n_transacciones_por_cat = 500

# Generar transacciones normales por categoría
transacciones = {}
ids_transaccion = {}

for i, cat in enumerate(categorias):
    media, std = params_categorias[cat]
    
    # Generar montos normales
    montos = np.random.normal(media, std, n_transacciones_por_cat)
    montos = np.maximum(montos, 10)  # Mínimo $10
    
    # Inyectar anomalías (aproximadamente 3-5% del total)
    n_anomalias_altas = np.random.randint(8, 15)
    n_anomalias_bajas = np.random.randint(5, 10)
    
    # Anomalías altas (montos sospechosamente grandes)
    indices_altas = np.random.choice(n_transacciones_por_cat, n_anomalias_altas, replace=False)
    montos[indices_altas] = media + np.random.uniform(4, 8, n_anomalias_altas) * std
    
    # Anomalías bajas (posibles pruebas de tarjeta)
    indices_bajas = np.random.choice(
        [i for i in range(n_transacciones_por_cat) if i not in indices_altas],
        n_anomalias_bajas, replace=False
    )
    montos[indices_bajas] = np.random.uniform(1, 15, n_anomalias_bajas)
    
    transacciones[cat] = montos
    ids_transaccion[cat] = np.arange(i * 1000 + 1, i * 1000 + n_transacciones_por_cat + 1)

# Crear arrays consolidados
# Array 2D: (categorías, transacciones)
montos_matriz = np.array([transacciones[cat] for cat in categorias])

# Arrays 1D para análisis global
todos_montos = np.concatenate([transacciones[cat] for cat in categorias])
todas_categorias = np.concatenate([[cat] * n_transacciones_por_cat for cat in categorias])
todos_ids = np.concatenate([ids_transaccion[cat] for cat in categorias])

print("╔══════════════════════════════════════════════════════════════════╗")
print("║              DATOS DE TRANSACCIONES GENERADOS                    ║")
print("╠══════════════════════════════════════════════════════════════════╣")
print(f"║  📊 montos_matriz    : shape {montos_matriz.shape}                      ║")
print(f"║     (filas=categorías, columnas=transacciones)                  ║")
print(f"║                                                                  ║")
print(f"║  💳 todos_montos     : {len(todos_montos):,} transacciones totales          ║")
print(f"║  🏷️  todas_categorias : {len(todas_categorias):,} etiquetas                    ║")
print(f"║  🔢 todos_ids        : {len(todos_ids):,} identificadores                ║")
print("╚══════════════════════════════════════════════════════════════════╝")

print("\n📍 Categorías disponibles:")
for i, cat in enumerate(categorias):
    media, std = params_categorias[cat]
    print(f"   {i}: {cat:20s} (μ=${media:,}, σ=${std})")

╔══════════════════════════════════════════════════════════════════╗
║              DATOS DE TRANSACCIONES GENERADOS                    ║
╠══════════════════════════════════════════════════════════════════╣
║  📊 montos_matriz    : shape (5, 500)                      ║
║     (filas=categorías, columnas=transacciones)                  ║
║                                                                  ║
║  💳 todos_montos     : 2,500 transacciones totales          ║
║  🏷️  todas_categorias : 2,500 etiquetas                    ║
║  🔢 todos_ids        : 2,500 identificadores                ║
╚══════════════════════════════════════════════════════════════════╝

📍 Categorías disponibles:
   0: Supermercados        (μ=$800, σ=$400)
   1: Restaurantes         (μ=$350, σ=$150)
   2: Gasolineras          (μ=$700, σ=$250)
   3: Tiendas_Online       (μ=$1,200, σ=$800)
   4: Entretenimiento      (μ=$200, σ=$100)


---

## 🏋️ PARTE 1: Análisis Estadístico por Categoría (30 puntos)

### Ejercicio 1.1: Estadísticas Descriptivas (10 puntos)

Calcula las estadísticas básicas para cada categoría de transacción.

In [5]:
# ═══════════════════════════════════════════════════════════════════
#            EJERCICIO 1.1: ESTADÍSTICAS POR CATEGORÍA
# ═══════════════════════════════════════════════════════════════════


# TODO: Para cada categoría, calcula y almacena:
# - Media
# - Mediana
# - Desviación estándar
# - Mínimo y Máximo

# ESTADÍSTICAS DESCRIPTIVAS POR CATEGORÍA (VERSIÓN VECTORIZADA)
print("ESTADÍSTICAS DESCRIPTIVAS POR CATEGORÍA")
print("═" * 80)

# Al especificar axis=1, NumPy calcula la estadística para cada fila (categoría) 
# de forma simultánea, sin necesidad de un bucle 'for'.
medias = np.mean(montos_matriz, axis=1)
medianas = np.median(montos_matriz, axis=1)
stds = np.std(montos_matriz, axis=1)
minimos = np.min(montos_matriz, axis=1)
maximos = np.max(montos_matriz, axis=1)

# Mostrar resultados en tabla
print(f"\n{'Categoría':<20} {'Media':>12} {'Mediana':>12} {'Std':>12} {'Mín':>10} {'Máx':>10}")
print("─" * 80)

# NOTA: En este punto, el único 'for' se usa estrictamente para 
# el formateo de impresión, NO para los cálculos.
for i, cat in enumerate(categorias):
    print(f"{cat:<20} ${medias[i]:>10,.2f} ${medianas[i]:>10,.2f} ${stds[i]:>10,.2f} ${minimos[i]:>8,.2f} ${maximos[i]:>8,.2f}")

ESTADÍSTICAS DESCRIPTIVAS POR CATEGORÍA
════════════════════════════════════════════════════════════════════════════════

Categoría                   Media      Mediana          Std        Mín        Máx
────────────────────────────────────────────────────────────────────────────────
Supermercados        $    853.56 $    797.39 $    576.21 $    1.32 $3,875.95
Restaurantes         $    362.13 $    352.39 $    203.49 $    2.61 $1,543.27
Gasolineras          $    722.50 $    688.70 $    349.05 $    1.13 $2,676.60
Tiendas_Online       $  1,337.98 $  1,158.48 $  1,122.47 $    1.46 $7,131.76
Entretenimiento      $    209.49 $    196.58 $    127.72 $    1.14 $  981.56


### Ejercicio 1.2: Cuartiles e IQR (10 puntos)

Calcula los cuartiles y el rango intercuartílico para cada categoría.

In [6]:
# ═══════════════════════════════════════════════════════════════════
#                 EJERCICIO 1.2: CUARTILES E IQR
# ═══════════════════════════════════════════════════════════════════

#  CUARTILES E IQR POR CATEGORÍA (VERSIÓN VECTORIZADA)
print("CUARTILES E IQR POR CATEGORÍA")
print("═" * 80)

# Calculamos los percentiles 25, 50 y 75 para toda la matriz de una sola vez
# axis=1 indica que el cálculo se hace por fila (categoría)
Q1_arr, Q2_arr, Q3_arr = np.percentile(montos_matriz, [25, 50, 75], axis=1)

# El IQR es la diferencia entre Q3 y Q1, se calcula vectorizadamente
IQR_arr = Q3_arr - Q1_arr

# Mostrar resultados
print(f"\n{'Categoría':<20} {'Q1 (25%)':>12} {'Q2 (50%)':>12} {'Q3 (75%)':>12} {'IQR':>12}")
print("─" * 72)

for i, cat in enumerate(categorias):
    print(f"{cat:<20} ${Q1_arr[i]:>10,.2f} ${Q2_arr[i]:>10,.2f} ${Q3_arr[i]:>10,.2f} ${IQR_arr[i]:>10,.2f}")

CUARTILES E IQR POR CATEGORÍA
════════════════════════════════════════════════════════════════════════════════

Categoría                Q1 (25%)     Q2 (50%)     Q3 (75%)          IQR
────────────────────────────────────────────────────────────────────────
Supermercados        $    510.27 $    797.39 $  1,103.41 $    593.14
Restaurantes         $    242.13 $    352.39 $    449.41 $    207.29
Gasolineras          $    511.29 $    688.70 $    881.19 $    369.90
Tiendas_Online       $    663.81 $  1,158.48 $  1,809.95 $  1,146.14
Entretenimiento      $    133.53 $    196.58 $    268.68 $    135.15


### Ejercicio 1.3: Límites para Outliers (10 puntos)

Calcula los límites inferior y superior para detectar outliers usando el método IQR.

In [7]:
# ═══════════════════════════════════════════════════════════════════
#               EJERCICIO 1.3: LÍMITES PARA OUTLIERS
# ═══════════════════════════════════════════════════════════════════

# LÍMITES PARA DETECCIÓN DE OUTLIERS (Método IQR - Vectorizado)
print("LÍMITES PARA DETECCIÓN DE OUTLIERS (Método IQR)")
print("═" * 80)

# Factor IQR estándar
FACTOR_IQR = 1.5

# TODO: Calcula los límites usando Q1, Q3 e IQR de forma vectorizada
# Al ser operaciones entre arrays, se calculan todas las categorías a la vez
limites_inf = Q1_arr - (FACTOR_IQR * IQR_arr)
limites_sup = Q3_arr + (FACTOR_IQR * IQR_arr)

# Mostrar resultados
print(f"\n{'Categoría':<20} {'Límite Inf':>15} {'Límite Sup':>15} {'Rango Válido':>20}")
print("─" * 75)

for i, cat in enumerate(categorias):
    # El límite inferior no puede ser negativo para montos, 
    # pero para el cálculo mantenemos la lógica matemática:
    lim_inf_real = max(0, limites_inf[i])
    rango = f"${lim_inf_real:,.0f} - ${limites_sup[i]:,.0f}"
    print(f"{cat:<20} ${limites_inf[i]:>13,.2f} ${limites_sup[i]:>13,.2f} {rango:>20}")

LÍMITES PARA DETECCIÓN DE OUTLIERS (Método IQR)
════════════════════════════════════════════════════════════════════════════════

Categoría                 Límite Inf      Límite Sup         Rango Válido
───────────────────────────────────────────────────────────────────────────
Supermercados        $      -379.45 $     1,993.12          $0 - $1,993
Restaurantes         $       -68.81 $       760.35            $0 - $760
Gasolineras          $       -43.56 $     1,436.03          $0 - $1,436
Tiendas_Online       $    -1,055.40 $     3,529.16          $0 - $3,529
Entretenimiento      $       -69.19 $       471.40            $0 - $471


---

## 🏋️ PARTE 2: Detección de Outliers con IQR (25 puntos)

### Ejercicio 2.1: Identificar Outliers por Categoría (15 puntos)

Detecta las transacciones anómalas en cada categoría usando el método IQR.

In [8]:
# ═══════════════════════════════════════════════════════════════════
#            EJERCICIO 2.1: DETECCIÓN DE OUTLIERS CON IQR
# ═══════════════════════════════════════════════════════════════════

# DETECCIÓN DE TRANSACCIONES ANÓMALAS (Método IQR - Vectorizado)
print(" DETECCIÓN DE TRANSACCIONES ANÓMALAS (Método IQR)")
print("═" * 80)

# 1. Creamos las máscaras para TODA la matriz usando broadcasting
# limites_inf y limites_sup tienen shape (5,), montos_matriz tiene (5, 500)
# Usamos .reshape(-1, 1) para que los límites se comparen fila a fila correctamente
mascara_outliers = (montos_matriz < limites_inf.reshape(-1, 1)) | (montos_matriz > limites_sup.reshape(-1, 1))

# 2. Resumen estadístico vectorizado
n_outliers_iqr = np.sum(mascara_outliers, axis=1) # Suma por fila
n_inf = np.sum(montos_matriz < limites_inf.reshape(-1, 1), axis=1)
n_sup = np.sum(montos_matriz > limites_sup.reshape(-1, 1), axis=1)

# 3. Almacenamos en el diccionario (el bucle aquí es solo para organizar la salida)
outliers_iqr = {}
for i, cat in enumerate(categorias):
    outliers_iqr[cat] = {
        'ids': ids_transaccion[cat][mascara_outliers[i]],
        'montos': montos_matriz[i][mascara_outliers[i]],
        'n_total': n_outliers_iqr[i],
        'n_inferiores': n_inf[i],
        'n_superiores': n_sup[i]
    }

# Mostrar resumen
print(f"\n{'Categoría':<20} {'Total Trans.':>12} {'Outliers':>10} {'% Anomalías':>12} {'Inf.':>8} {'Sup.':>8}")
print("─" * 75)

for i, cat in enumerate(categorias):
    pct = (n_outliers_iqr[i] / n_transacciones_por_cat) * 100
    info = outliers_iqr[cat]
    print(f"{cat:<20} {n_transacciones_por_cat:>12,} {info['n_total']:>10} {pct:>11.1f}% {info['n_inferiores']:>8} {info['n_superiores']:>8}")

print(f"\n Total de outliers detectados: {np.sum(n_outliers_iqr)}")

 DETECCIÓN DE TRANSACCIONES ANÓMALAS (Método IQR)
════════════════════════════════════════════════════════════════════════════════

Categoría            Total Trans.   Outliers  % Anomalías     Inf.     Sup.
───────────────────────────────────────────────────────────────────────────
Supermercados                 500         15         3.0%        0       15
Restaurantes                  500         16         3.2%        0       16
Gasolineras                   500         13         2.6%        0       13
Tiendas_Online                500         14         2.8%        0       14
Entretenimiento               500         13         2.6%        0       13

 Total de outliers detectados: 71


### Ejercicio 2.2: Análisis de Outliers Detectados (10 puntos)

Analiza las características de los outliers detectados.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#              EJERCICIO 2.2: ANÁLISIS DE OUTLIERS IQR
# ═══════════════════════════════════════════════════════════════════

# TODO: Calcula estadísticas de los outliers
        # info['montos'] ya contiene los valores que cumplen la condición de outlier
        montos_out = info['montos']
            monto_min_outlier = np.min(montos_out)
        monto_max_outlier = np.max(montos_out)
        monto_promedio_outlier = np.mean(montos_out)
        
        print(f"\n🏷️  {cat}")
        print(f"   Outliers detectados: {info['n_total']}")
        print(f"   Monto mínimo outlier: ${monto_min_outlier:,.2f}")
        print(f"   Monto máximo outlier: ${monto_max_outlier:,.2f}")
        print(f"   Monto promedio outlier: ${monto_promedio_outlier:,.2f}")
        
        # Mostrar los 3 outliers más extremos (más altos)
        if info['n_superiores'] > 0:
            # np.argsort devuelve los índices que ordenarían el array
            # [::-1] invierte el orden para obtener los valores más grandes primero
            idx_ordenados = np.argsort(montos_out)[::-1] 
            print(f"   Top 3 montos más altos:")
            for j in range(min(3, len(idx_ordenados))):
                idx = idx_ordenados[j]
                print(f"     - ID {info['ids'][idx]}: ${montos_out[idx]:,.2f}")

---

## 🏋️ PARTE 3: Detección de Outliers con Z-Score (25 puntos)

### Ejercicio 3.1: Calcular Z-Scores (10 puntos)

Calcula el Z-Score para cada transacción.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#               EJERCICIO 3.1: CÁLCULO DE Z-SCORES
# ═══════════════════════════════════════════════════════════════════

print("📐 CÁLCULO DE Z-SCORES POR CATEGORÍA")
print("═" * 80)

# Umbral para considerar outlier
UMBRAL_ZSCORE = 3

# Matriz para almacenar z-scores
zscores_matriz = np.zeros_like(montos_matriz)

for i, cat in enumerate(categorias):
    datos = montos_matriz[i]
    
    # TODO: Calcula el Z-Score para cada transacción
    # Fórmula: z = (x - media) / std
    
    media_cat = ___
    std_cat = ___
    zscores_matriz[i] = ___

# Verificar cálculo mostrando estadísticas de z-scores
print(f"\n{'Categoría':<20} {'Media Z':>10} {'Std Z':>10} {'Min Z':>10} {'Max Z':>10}")
print("─" * 65)

for i, cat in enumerate(categorias):
    zs = zscores_matriz[i]
    print(f"{cat:<20} {np.mean(zs):>10.4f} {np.std(zs):>10.4f} {np.min(zs):>10.2f} {np.max(zs):>10.2f}")

print(f"\n💡 Nota: La media de Z-scores debe ser ~0 y la std ~1")

### Ejercicio 3.2: Detectar Outliers con Z-Score (15 puntos)

Identifica outliers donde |Z-Score| > 3.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#          EJERCICIO 3.2: DETECCIÓN DE OUTLIERS CON Z-SCORE
# ═══════════════════════════════════════════════════════════════════

print(f"🔍 DETECCIÓN DE OUTLIERS CON Z-SCORE (umbral = {UMBRAL_ZSCORE})")
print("═" * 80)

# Diccionario para almacenar resultados
outliers_zscore = {}
n_outliers_zscore = np.zeros(n_categorias, dtype=int)

for i, cat in enumerate(categorias):
    datos = montos_matriz[i]
    zscores = zscores_matriz[i]
    ids = ids_transaccion[cat]
    
    # TODO: Crea una máscara para outliers donde |z-score| > umbral
    mascara_outliers_z = ___
    
    # Clasificar por tipo
    mascara_z_neg = zscores < -UMBRAL_ZSCORE  # Muy bajos
    mascara_z_pos = zscores > UMBRAL_ZSCORE   # Muy altos
    
    outliers_zscore[cat] = {
        'ids': ids[mascara_outliers_z],
        'montos': datos[mascara_outliers_z],
        'zscores': zscores[mascara_outliers_z],
        'n_total': np.sum(mascara_outliers_z),
        'n_bajos': np.sum(mascara_z_neg),
        'n_altos': np.sum(mascara_z_pos)
    }
    n_outliers_zscore[i] = np.sum(mascara_outliers_z)

# Mostrar resumen
print(f"\n{'Categoría':<20} {'Total Trans.':>12} {'Outliers':>10} {'% Anomalías':>12} {'Z<-3':>8} {'Z>3':>8}")
print("─" * 75)

for i, cat in enumerate(categorias):
    pct = (n_outliers_zscore[i] / n_transacciones_por_cat) * 100
    info = outliers_zscore[cat]
    print(f"{cat:<20} {n_transacciones_por_cat:>12,} {info['n_total']:>10} {pct:>11.1f}% {info['n_bajos']:>8} {info['n_altos']:>8}")

print(f"\n📊 Total de outliers detectados (Z-Score): {np.sum(n_outliers_zscore)}")

---

## 🏋️ PARTE 4: Comparación y Reporte Final (20 puntos)

### Ejercicio 4.1: Comparar Métodos (10 puntos)

Compara los resultados de ambos métodos de detección.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#               EJERCICIO 4.1: COMPARACIÓN DE MÉTODOS
# ═══════════════════════════════════════════════════════════════════

print("🔄 COMPARACIÓN DE MÉTODOS DE DETECCIÓN")
print("═" * 80)

# TODO: Calcula métricas de comparación

# Total de outliers por método
total_iqr = ___
total_zscore = ___

print(f"\n📊 RESUMEN GLOBAL:")
print(f"   Método IQR:     {total_iqr} outliers detectados")
print(f"   Método Z-Score: {total_zscore} outliers detectados")

# Comparación por categoría
print(f"\n{'Categoría':<20} {'IQR':>10} {'Z-Score':>10} {'Diferencia':>12} {'Coincidencia':>15}")
print("─" * 72)

for i, cat in enumerate(categorias):
    n_iqr = n_outliers_iqr[i]
    n_zs = n_outliers_zscore[i]
    diff = n_iqr - n_zs
    
    # TODO: Calcula cuántos outliers coinciden en ambos métodos
    # (están en ambas listas de outliers)
    ids_iqr = set(outliers_iqr[cat]['ids'])
    ids_zscore = set(outliers_zscore[cat]['ids'])
    coincidentes = len(ids_iqr & ids_zscore)  # Intersección
    
    print(f"{cat:<20} {n_iqr:>10} {n_zs:>10} {diff:>+12} {coincidentes:>15}")

### Ejercicio 4.2: Reporte de Transacciones Sospechosas (10 puntos)

Genera un reporte final de las transacciones más sospechosas.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#           EJERCICIO 4.2: REPORTE DE TRANSACCIONES SOSPECHOSAS
# ═══════════════════════════════════════════════════════════════════

print("╔══════════════════════════════════════════════════════════════════════════╗")
print("║                                                                          ║")
print("║        🚨 SECUREBANK - REPORTE DE TRANSACCIONES SOSPECHOSAS 🚨           ║")
print("║                                                                          ║")
print("╠══════════════════════════════════════════════════════════════════════════╣")

# Identificar transacciones detectadas por AMBOS métodos (alta confianza)
print("║                                                                          ║")
print("║  ⚠️  ALTA PRIORIDAD (detectadas por ambos métodos)                       ║")
print("║  ─────────────────────────────────────────────────────────────────────   ║")

total_alta_prioridad = 0

for cat in categorias:
    ids_iqr = set(outliers_iqr[cat]['ids'])
    ids_zscore = set(outliers_zscore[cat]['ids'])
    
    # TODO: Encuentra las transacciones que están en AMBOS conjuntos
    ids_ambos = ___
    
    if len(ids_ambos) > 0:
        total_alta_prioridad += len(ids_ambos)
        
        # Obtener montos de estas transacciones
        for trans_id in list(ids_ambos)[:3]:  # Mostrar máximo 3
            idx = np.where(ids_transaccion[cat] == trans_id)[0][0]
            monto = montos_matriz[categorias.index(cat), idx]
            zscore = zscores_matriz[categorias.index(cat), idx]
            print(f"║    ID {trans_id}: {cat:15s} ${monto:>10,.2f} (Z={zscore:+.2f}){'':>10}║")

print("║                                                                          ║")
print(f"║  📊 RESUMEN EJECUTIVO                                                    ║")
print("║  ─────────────────────────────────────────────────────────────────────   ║")

# TODO: Calcula estadísticas finales
total_transacciones = ___
total_outliers_unicos = ___  # Unión de ambos métodos
pct_anomalias = (total_outliers_unicos / total_transacciones) * 100

print(f"║    Total transacciones analizadas:    {total_transacciones:>6,}{'':>25}║")
print(f"║    Transacciones sospechosas:         {total_outliers_unicos:>6,} ({pct_anomalias:.1f}%){'':>17}║")
print(f"║    Alta prioridad (ambos métodos):    {total_alta_prioridad:>6,}{'':>25}║")
print("║                                                                          ║")
print("╚══════════════════════════════════════════════════════════════════════════╝")

---

## 🎯 EJERCICIO BONUS: Análisis de Correlación (10 puntos)

Analiza si existe correlación entre categorías en los patrones de gasto.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#                  EJERCICIO BONUS: CORRELACIÓN
# ═══════════════════════════════════════════════════════════════════

print("📈 ANÁLISIS DE CORRELACIÓN ENTRE CATEGORÍAS")
print("═" * 70)

# TODO: Calcula la matriz de correlación entre categorías
# Usa montos_matriz donde cada fila es una categoría

matriz_correlacion = ___

# Mostrar matriz de correlación
print(f"\n{'':>18}", end='')
for cat in categorias:
    print(f"{cat[:8]:>10}", end='')
print()
print("─" * 70)

for i, cat in enumerate(categorias):
    print(f"{cat:<18}", end='')
    for j in range(n_categorias):
        valor = matriz_correlacion[i, j]
        if i == j:
            print(f"{'1.00':>10}", end='')
        else:
            print(f"{valor:>10.3f}", end='')
    print()

# Encontrar las correlaciones más fuertes (excluyendo diagonal)
print(f"\n🔗 CORRELACIONES MÁS FUERTES:")
print("─" * 50)

# TODO: Identifica el par de categorías con mayor correlación
# (excluyendo la diagonal)
max_corr = 0
par_max = ('', '')

for i in range(n_categorias):
    for j in range(i+1, n_categorias):
        if abs(matriz_correlacion[i, j]) > abs(max_corr):
            max_corr = matriz_correlacion[i, j]
            par_max = (categorias[i], categorias[j])

print(f"   Mayor correlación: {par_max[0]} ↔ {par_max[1]}")
print(f"   Valor: {max_corr:.4f}")

---

## ✅ Checklist de Entrega

Antes de entregar, verifica que:

| # | Criterio | Puntos |
|---|----------|--------|
| 1 | Parte 1: Análisis Estadístico por Categoría | 30 |
| 2 | Parte 2: Detección de Outliers con IQR | 25 |
| 3 | Parte 3: Detección de Outliers con Z-Score | 25 |
| 4 | Parte 4: Comparación y Reporte Final | 20 |
| 5 | BONUS: Análisis de Correlación | +10 |
| | **Total posible** | **110** |

### 📝 Notas Importantes

- Usa operaciones vectorizadas de NumPy (evita loops donde sea posible)
- Verifica que tus Z-Scores tengan media ~0 y std ~1
- El código debe ejecutarse sin errores de principio a fin
- Los outliers detectados deben ser razonables (3-5% típicamente)

---

### 🏅 Criterios de Evaluación

```
RÚBRICA DE CALIFICACIÓN
═══════════════════════════════════════════════════════

✓ Cálculos estadísticos correctos         → 40%
✓ Detección de outliers precisa           → 30%
✓ Comparación de métodos coherente        → 15%
✓ Código limpio y vectorizado             → 10%
✓ Interpretaciones razonables             → 5%

═══════════════════════════════════════════════════════
```

---

*SecureBank Fraud Detection - Reto desarrollado para Programación para Ciencia de Datos, IPN 2026*